In [1]:
# ============================================================
# PHASE 2.1 — GENOMIC HANDCRAFTED FEATURE BASELINE
# TSS-proximal regulatory DNA: 2kb upstream + 500bp downstream
# ============================================================

import itertools
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    VotingClassifier,
    StackingClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    make_scorer
)

import joblib

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

In [3]:
# ============================================================
# PATHS
# ============================================================
import os
from pathlib import Path
import pandas as pd
from google.colab import drive

try:
    drive.mount('/content/drive')
except ValueError:
    print("Drive đã được kết nối từ trước.")

PROJECT_DIR = Path("/content/drive/MyDrive/Project_Protein")

PHASE2_DIR = PROJECT_DIR / "model" / "phase2_genomic_regulatory_baseline"

SPLIT_DIR = PHASE2_DIR / "splits"
FEATURE_DIR = PHASE2_DIR / "features"
RESULT_DIR = PHASE2_DIR / "results"
MODEL_DIR = PHASE2_DIR / "models"
REPORT_DIR = PHASE2_DIR / "reports"

for folder in [FEATURE_DIR, RESULT_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

train_path = SPLIT_DIR / "train_genomic_regulatory_v1.csv"
val_path = SPLIT_DIR / "val_genomic_regulatory_v1.csv"
test_path = SPLIT_DIR / "test_genomic_regulatory_v1.csv"

assert train_path.exists(), train_path
assert val_path.exists(), val_path
assert test_path.exists(), test_path

print("Phase 2 dir:", PHASE2_DIR)

Mounted at /content/drive
Phase 2 dir: /content/drive/MyDrive/Project_Protein/model/phase2_genomic_regulatory_baseline


In [4]:
# ============================================================
# LOAD PHASE 2.0 SPLITS
# ============================================================

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain labels:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation labels:")
print(val_df["label"].value_counts().sort_index())

print("\nTest labels:")
print(test_df["label"].value_counts().sort_index())

display(train_df.head(3))

Train: (1264, 20)
Validation: (271, 20)
Test: (271, 20)

Train labels:
label
0    632
1    632
Name: count, dtype: int64

Validation labels:
label
0    135
1    136
Name: count, dtype: int64

Test labels:
label
0    136
1    135
Name: count, dtype: int64


,gene_id,gene_symbol,gene_name,label,dataset_role,ensembl_gene_id,ensembl_gene_name,gene_type,chromosome_or_scaffold,gene_start_bp,gene_end_bp,strand,tss_sequence_orientation,regulatory_sequence_source,regulatory_sequence_scope,regulatory_sequence_version,regulatory_sequence,regulatory_sequence_length,n_N_bases,invalid_dna_chars
0,ENSG00000149201,CCDC81,CCDC81,0,negative,ENSG00000149201,CCDC81,protein_coding,11,86374736.0,86423109.0,1.0,forward_gene_orientation,Ensembl Homo sapiens GRCh38 primary assembly FASTA,TSS-proximal window: 2000 bp upstream + 500 bp downstream,TSS_Proximal_2kbUp_500bpDown_NuclearBalanced_v1,ACTCTACCTGTACCTCTCTCTCTTCCTCTCTTTGGTAAATTATCTTAAATATTATAAAGTAATTCATGCCCTTGTTAAACAGTAAATAAAGGAGAAAAGTGAATATAAAAGTTCTT...,2500,0,NaN
1,ENSG00000127324,TSPAN8,tetraspanin 8,1,positive,ENSG00000127324,TSPAN8,protein_coding,12,71124454.0,71441898.0,-1.0,reverse_complement_gene_orientation,Ensembl Homo sapiens GRCh38 primary assembly FASTA,TSS-proximal window: 2000 bp upstream + 500 bp downstream,TSS_Proximal_2kbUp_500bpDown_NuclearBalanced_v1,GGAGGGACAAAATTCTGGAAAGTGGGTGGAGGACTTGTTTGCTGTGTGCTGTAAATGCTACTTTAAGGAATCTGAACTTTATCTTGTAGGTAAAAGGAAAATATCCTTAGATTGGG...,2500,0,NaN
2,ENSG00000170262,MRAP,MRAP,0,negative,ENSG00000170262,MRAP,protein_coding,21,32291813.0,32314784.0,1.0,forward_gene_orientation,Ensembl Homo sapiens GRCh38 primary assembly FASTA,TSS-proximal window: 2000 bp upstream + 500 bp downstream,TSS_Proximal_2kbUp_500bpDown_NuclearBalanced_v1,GGAGTGGTGAGTGAATGCGAAGGCCTAGGACATTACTGGACACTACTGTAGACTTTATAAGCACCACACTTAGCCTATACTAAATTTATTTATAAAATAAAGTAAATGTGCAATGA...,2500,0,NaN


In [5]:
# ============================================================
# BASIC SEQUENCE CHECK
# ============================================================

SEQ_COL = "regulatory_sequence"
LABEL_COL = "label"

for split_name, split_df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    print("=" * 80)
    print(split_name)
    print("n:", len(split_df))
    print("length unique:", split_df[SEQ_COL].str.len().unique()[:10])
    print("label counts:")
    print(split_df[LABEL_COL].value_counts().sort_index())

    assert split_df[SEQ_COL].notna().all()
    assert split_df[SEQ_COL].str.len().eq(2500).all()
    assert split_df[LABEL_COL].isin([0, 1]).all()

train
n: 1264
length unique: [2500]
label counts:
label
0    632
1    632
Name: count, dtype: int64
validation
n: 271
length unique: [2500]
label counts:
label
0    135
1    136
Name: count, dtype: int64
test
n: 271
length unique: [2500]
label counts:
label
0    136
1    135
Name: count, dtype: int64


In [6]:
# ============================================================
# GENOMIC FEATURE FUNCTIONS
# ============================================================

DNA_ALPHABET = "ACGT"


def safe_divide(a, b):
    return a / b if b != 0 else 0.0


def generate_kmers(k):
    return ["".join(p) for p in itertools.product(DNA_ALPHABET, repeat=k)]


KMER_LISTS = {
    3: generate_kmers(3),
    4: generate_kmers(4),
    5: generate_kmers(5)
}

print("Number of 3-mers:", len(KMER_LISTS[3]))
print("Number of 4-mers:", len(KMER_LISTS[4]))
print("Number of 5-mers:", len(KMER_LISTS[5]))

Number of 3-mers: 64
Number of 4-mers: 256
Number of 5-mers: 1024


In [7]:
def compute_basic_dna_features(seq, n_bins=10):
    """
    Compute global and positional DNA composition features.
    Sequence expected to be uppercase A/C/G/T only.
    """
    seq = str(seq).upper()
    L = len(seq)

    a = seq.count("A")
    c = seq.count("C")
    g = seq.count("G")
    t = seq.count("T")

    gc = g + c
    at = a + t

    cpg_count = seq.count("CG")

    # Expected CpG under independence:
    # observed/expected CpG = (CpG_count * L) / (C_count * G_count)
    cpg_oe = safe_divide(cpg_count * L, c * g)

    features = {
        "seq_length": L,

        "A_count": a,
        "C_count": c,
        "G_count": g,
        "T_count": t,

        "A_frac": safe_divide(a, L),
        "C_frac": safe_divide(c, L),
        "G_frac": safe_divide(g, L),
        "T_frac": safe_divide(t, L),

        "GC_count": gc,
        "AT_count": at,
        "GC_content": safe_divide(gc, L),
        "AT_content": safe_divide(at, L),

        "CpG_count": cpg_count,
        "CpG_density": safe_divide(cpg_count, L),
        "CpG_observed_expected": cpg_oe,

        "GC_skew": safe_divide(g - c, g + c),
        "AT_skew": safe_divide(a - t, a + t),

        "purine_frac_AG": safe_divide(a + g, L),
        "pyrimidine_frac_CT": safe_divide(c + t, L),
    }

    # Positional GC bins
    bin_size = L // n_bins

    for i in range(n_bins):
        start = i * bin_size
        end = (i + 1) * bin_size if i < n_bins - 1 else L
        sub = seq[start:end]
        sub_L = len(sub)
        sub_gc = sub.count("G") + sub.count("C")

        features[f"GC_content_bin_{i+1:02d}"] = safe_divide(sub_gc, sub_L)

    # Promoter/TSS-specific coarse regions
    # Input window: 2000bp upstream + 500bp downstream.
    # Position 0-1999 upstream, 2000 is TSS boundary approximately.
    upstream = seq[:2000]
    downstream = seq[2000:]
    tss_near = seq[1750:2250]  # -250 to +250 around TSS

    features["GC_content_upstream_2kb"] = safe_divide(
        upstream.count("G") + upstream.count("C"),
        len(upstream)
    )

    features["GC_content_downstream_500bp"] = safe_divide(
        downstream.count("G") + downstream.count("C"),
        len(downstream)
    )

    features["GC_content_tss_near_500bp"] = safe_divide(
        tss_near.count("G") + tss_near.count("C"),
        len(tss_near)
    )

    features["CpG_count_upstream_2kb"] = upstream.count("CG")
    features["CpG_count_downstream_500bp"] = downstream.count("CG")
    features["CpG_count_tss_near_500bp"] = tss_near.count("CG")

    return features

In [8]:
def compute_kmer_frequencies(seq, k):
    """
    Compute normalized k-mer frequencies for DNA sequence.
    Uses overlapping k-mers.
    """
    seq = str(seq).upper()
    kmers = KMER_LISTS[k]

    counts = dict.fromkeys(kmers, 0)

    total = len(seq) - k + 1

    if total <= 0:
        return {f"k{k}_{mer}": 0.0 for mer in kmers}

    for i in range(total):
        mer = seq[i:i+k]
        if mer in counts:
            counts[mer] += 1

    return {
        f"k{k}_{mer}": counts[mer] / total
        for mer in kmers
    }

In [9]:
def extract_genomic_features_for_df(
    df,
    sequence_col="regulatory_sequence",
    include_kmers=(3, 4),
    n_bins=10
):
    """
    Extract handcrafted genomic features.
    """
    feature_records = []

    for idx, row in df.iterrows():
        seq = row[sequence_col]

        record = {}

        basic_features = compute_basic_dna_features(seq, n_bins=n_bins)
        record.update(basic_features)

        for k in include_kmers:
            kmer_features = compute_kmer_frequencies(seq, k=k)
            record.update(kmer_features)

        feature_records.append(record)

    feature_df = pd.DataFrame(feature_records)

    return feature_df

In [10]:
# ============================================================
# EXTRACT BASIC FEATURES ONLY
# ============================================================

X_train_basic = extract_genomic_features_for_df(
    train_df,
    sequence_col=SEQ_COL,
    include_kmers=(),
    n_bins=10
)

X_val_basic = extract_genomic_features_for_df(
    val_df,
    sequence_col=SEQ_COL,
    include_kmers=(),
    n_bins=10
)

X_test_basic = extract_genomic_features_for_df(
    test_df,
    sequence_col=SEQ_COL,
    include_kmers=(),
    n_bins=10
)

print("Basic features:", X_train_basic.shape)
display(X_train_basic.head())

Basic features: (1264, 36)


,seq_length,A_count,C_count,G_count,T_count,A_frac,C_frac,G_frac,T_frac,GC_count,AT_count,GC_content,AT_content,CpG_count,CpG_density,CpG_observed_expected,GC_skew,AT_skew,purine_frac_AG,pyrimidine_frac_CT,GC_content_bin_01,GC_content_bin_02,GC_content_bin_03,GC_content_bin_04,GC_content_bin_05,GC_content_bin_06,GC_content_bin_07,GC_content_bin_08,GC_content_bin_09,GC_content_bin_10,GC_content_upstream_2kb,GC_content_downstream_500bp,GC_content_tss_near_500bp,CpG_count_upstream_2kb,CpG_count_downstream_500bp,CpG_count_tss_near_500bp
0,2500,753,531,492,724,0.3012,0.2124,0.1968,0.2896,1023,1477,0.4092,0.5908,42,0.0168,0.401911,-0.038123,0.019634,0.4980,0.5020,0.300,0.280,0.312,0.300,0.356,0.408,0.424,0.632,0.464,0.616,0.3765,0.540,0.548,25,17,26
1,2500,805,391,484,820,0.3220,0.1564,0.1936,0.3280,875,1625,0.3500,0.6500,13,0.0052,0.171736,0.106286,-0.009231,0.5156,0.4844,0.360,0.348,0.276,0.268,0.388,0.384,0.328,0.408,0.388,0.352,0.3450,0.370,0.398,7,6,4
2,2500,644,557,622,677,0.2576,0.2228,0.2488,0.2708,1179,1321,0.4716,0.5284,41,0.0164,0.295855,0.055131,-0.024981,0.5064,0.4936,0.364,0.460,0.456,0.456,0.508,0.476,0.428,0.564,0.584,0.420,0.4640,0.502,0.574,24,17,17
3,2500,477,719,682,622,0.1908,0.2876,0.2728,0.2488,1401,1099,0.5604,0.4396,148,0.0592,0.754551,-0.026410,-0.131938,0.4636,0.5364,0.484,0.516,0.460,0.392,0.392,0.564,0.584,0.712,0.752,0.748,0.5130,0.750,0.732,81,67,60
4,2500,532,705,765,498,0.2128,0.2820,0.3060,0.1992,1470,1030,0.5880,0.4120,107,0.0428,0.495990,0.040816,0.033010,0.5188,0.4812,0.544,0.580,0.556,0.480,0.592,0.568,0.564,0.704,0.688,0.604,0.5735,0.646,0.696,70,37,52


In [11]:
# ============================================================
# EXTRACT 3-MER FEATURES
# ============================================================

X_train_k3 = extract_genomic_features_for_df(
    train_df,
    sequence_col=SEQ_COL,
    include_kmers=(3,),
    n_bins=10
)

X_val_k3 = extract_genomic_features_for_df(
    val_df,
    sequence_col=SEQ_COL,
    include_kmers=(3,),
    n_bins=10
)

X_test_k3 = extract_genomic_features_for_df(
    test_df,
    sequence_col=SEQ_COL,
    include_kmers=(3,),
    n_bins=10
)

# k3-only columns
k3_cols = [c for c in X_train_k3.columns if c.startswith("k3_")]

print("K3 full with basic:", X_train_k3.shape)
print("K3-only cols:", len(k3_cols))

K3 full with basic: (1264, 100)
K3-only cols: 64


In [12]:
# ============================================================
# EXTRACT 4-MER FEATURES
# ============================================================

X_train_k4 = extract_genomic_features_for_df(
    train_df,
    sequence_col=SEQ_COL,
    include_kmers=(4,),
    n_bins=10
)

X_val_k4 = extract_genomic_features_for_df(
    val_df,
    sequence_col=SEQ_COL,
    include_kmers=(4,),
    n_bins=10
)

X_test_k4 = extract_genomic_features_for_df(
    test_df,
    sequence_col=SEQ_COL,
    include_kmers=(4,),
    n_bins=10
)

k4_cols = [c for c in X_train_k4.columns if c.startswith("k4_")]

print("K4 full with basic:", X_train_k4.shape)
print("K4-only cols:", len(k4_cols))

K4 full with basic: (1264, 292)
K4-only cols: 256


In [13]:
# ============================================================
# EXTRACT MAIN FEATURE SET: BASIC + K3 + K4
# ============================================================

X_train_main = extract_genomic_features_for_df(
    train_df,
    sequence_col=SEQ_COL,
    include_kmers=(3, 4),
    n_bins=10
)

X_val_main = extract_genomic_features_for_df(
    val_df,
    sequence_col=SEQ_COL,
    include_kmers=(3, 4),
    n_bins=10
)

X_test_main = extract_genomic_features_for_df(
    test_df,
    sequence_col=SEQ_COL,
    include_kmers=(3, 4),
    n_bins=10
)

print("Main genomic features:", X_train_main.shape)
display(X_train_main.head())

Main genomic features: (1264, 356)


,seq_length,A_count,C_count,G_count,T_count,A_frac,C_frac,G_frac,T_frac,GC_count,AT_count,GC_content,AT_content,CpG_count,CpG_density,CpG_observed_expected,GC_skew,AT_skew,purine_frac_AG,pyrimidine_frac_CT,GC_content_bin_01,GC_content_bin_02,GC_content_bin_03,GC_content_bin_04,GC_content_bin_05,GC_content_bin_06,GC_content_bin_07,GC_content_bin_08,GC_content_bin_09,GC_content_bin_10,GC_content_upstream_2kb,GC_content_downstream_500bp,GC_content_tss_near_500bp,CpG_count_upstream_2kb,CpG_count_downstream_500bp,CpG_count_tss_near_500bp,k3_AAA,k3_AAC,k3_AAG,k3_AAT,k3_ACA,k3_ACC,k3_ACG,k3_ACT,k3_AGA,k3_AGC,k3_AGG,k3_AGT,k3_ATA,k3_ATC,k3_ATG,k3_ATT,k3_CAA,k3_CAC,k3_CAG,k3_CAT,k3_CCA,k3_CCC,k3_CCG,k3_CCT,k3_CGA,k3_CGC,k3_CGG,k3_CGT,k3_CTA,k3_CTC,k3_CTG,k3_CTT,k3_GAA,k3_GAC,k3_GAG,k3_GAT,k3_GCA,k3_GCC,k3_GCG,k3_GCT,k3_GGA,k3_GGC,k3_GGG,k3_GGT,k3_GTA,k3_GTC,k3_GTG,k3_GTT,k3_TAA,k3_TAC,k3_TAG,k3_TAT,k3_TCA,k3_TCC,k3_TCG,k3_TCT,k3_TGA,k3_TGC,k3_TGG,k3_TGT,k3_TTA,k3_TTC,k3_TTG,k3_TTT,...,k4_GCTA,k4_GCTC,k4_GCTG,k4_GCTT,k4_GGAA,k4_GGAC,k4_GGAG,k4_GGAT,k4_GGCA,k4_GGCC,k4_GGCG,k4_GGCT,k4_GGGA,k4_GGGC,k4_GGGG,k4_GGGT,k4_GGTA,k4_GGTC,k4_GGTG,k4_GGTT,k4_GTAA,k4_GTAC,k4_GTAG,k4_GTAT,k4_GTCA,k4_GTCC,k4_GTCG,k4_GTCT,k4_GTGA,k4_GTGC,k4_GTGG,k4_GTGT,k4_GTTA,k4_GTTC,k4_GTTG,k4_GTTT,k4_TAAA,k4_TAAC,k4_TAAG,k4_TAAT,k4_TACA,k4_TACC,k4_TACG,k4_TACT,k4_TAGA,k4_TAGC,k4_TAGG,k4_TAGT,k4_TATA,k4_TATC,k4_TATG,k4_TATT,k4_TCAA,k4_TCAC,k4_TCAG,k4_TCAT,k4_TCCA,k4_TCCC,k4_TCCG,k4_TCCT,k4_TCGA,k4_TCGC,k4_TCGG,k4_TCGT,k4_TCTA,k4_TCTC,k4_TCTG,k4_TCTT,k4_TGAA,k4_TGAC,k4_TGAG,k4_TGAT,k4_TGCA,k4_TGCC,k4_TGCG,k4_TGCT,k4_TGGA,k4_TGGC,k4_TGGG,k4_TGGT,k4_TGTA,k4_TGTC,k4_TGTG,k4_TGTT,k4_TTAA,k4_TTAC,k4_TTAG,k4_TTAT,k4_TTCA,k4_TTCC,k4_TTCG,k4_TTCT,k4_TTGA,k4_TTGC,k4_TTGG,k4_TTGT,k4_TTTA,k4_TTTC,k4_TTTG,k4_TTTT
0,2500,753,531,492,724,0.3012,0.2124,0.1968,0.2896,1023,1477,0.4092,0.5908,42,0.0168,0.401911,-0.038123,0.019634,0.4980,0.5020,0.300,0.280,0.312,0.300,0.356,0.408,0.424,0.632,0.464,0.616,0.3765,0.540,0.548,25,17,26,0.041633,0.014812,0.020416,0.028022,0.017214,0.011209,0.004003,0.013611,0.017614,0.015212,0.015212,0.014812,0.025220,0.015212,0.018415,0.028423,0.020416,0.010809,0.018415,0.019616,0.019215,0.013211,0.003603,0.020416,0.004003,0.005204,0.003603,0.004003,0.012810,0.020416,0.018815,0.017614,0.020817,0.007206,0.015212,0.009608,0.015612,0.015212,0.003603,0.012410,0.016413,0.008807,0.017614,0.009207,0.014812,0.006805,0.012010,0.011609,0.022018,0.012810,0.008807,0.030024,0.017614,0.016813,0.005604,0.023219,0.014812,0.017614,0.015612,0.017214,0.020817,0.020817,0.016013,0.030024,...,0.002403,0.002803,0.005206,0.002002,0.006408,0.001602,0.005206,0.003204,0.003204,0.001602,0.001201,0.002803,0.004405,0.002403,0.006408,0.004405,0.002803,0.002403,0.002803,0.001201,0.004806,0.002403,0.002002,0.005607,0.004005,0.001201,0.000400,0.001201,0.002403,0.004005,0.003604,0.002002,0.002002,0.002403,0.002403,0.004806,0.009211,0.001201,0.004005,0.007609,0.005607,0.002403,0.001201,0.003604,0.002002,0.004005,0.001201,0.001602,0.009612,0.004806,0.006007,0.009612,0.006408,0.002803,0.004005,0.004005,0.006808,0.004806,0.000400,0.004806,0.002002,0.002403,0.000400,0.000801,0.005206,0.006007,0.004806,0.007209,0.006408,0.003204,0.002403,0.002803,0.006408,0.007209,0.000400,0.003604,0.006408,0.002403,0.004005,0.002803,0.006408,0.001602,0.004806,0.004405,0.006808,0.003204,0.002002,0.008811,0.005607,0.008410,0.000801,0.006007,0.004005,0.003604,0.004005,0.004405,0.007209,0.004806,0.006808,0.011213
1,2500,805,391,484,820,0.3220,0.1564,0.1936,0.3280,875,1625,0.3500,0.6500,13,0.0052,0.171736,0.106286,-0.009231,0.5156,0.4844,0.360,0.348,0.276,0.268,0.388,0.384,0.328,0.408,0.388,0.352,0.3450,0.370,0.398,7,6,4,0.043235,0.015212,0.020416,0.036429,0.019215,0.005604,0.002002,0.013211,0.024019,0.011609,0.018815,0.017214,0.025620,0.014412,0.017614,0.037630,0.019215,0.007606,0.018014,0.015212,0.011209,0.005204,0.001201,0.014812,0.001601,0.000801,0.002002,0.000801,0.014011,0.008407,0.016413,0.019215,0.026021,0.0068

In [14]:
# ============================================================
# OPTIONAL HIGH-DIM FEATURE SET: BASIC + K3 + K4 + K5
# Run this if time is okay.
# ============================================================

RUN_K5 = True

if RUN_K5:
    X_train_k345 = extract_genomic_features_for_df(
        train_df,
        sequence_col=SEQ_COL,
        include_kmers=(3, 4, 5),
        n_bins=10
    )

    X_val_k345 = extract_genomic_features_for_df(
        val_df,
        sequence_col=SEQ_COL,
        include_kmers=(3, 4, 5),
        n_bins=10
    )

    X_test_k345 = extract_genomic_features_for_df(
        test_df,
        sequence_col=SEQ_COL,
        include_kmers=(3, 4, 5),
        n_bins=10
    )

    print("K3+K4+K5 features:", X_train_k345.shape)

K3+K4+K5 features: (1264, 1380)


In [15]:
# ============================================================
# SAVE FEATURE MATRICES
# ============================================================

def save_feature_set(name, X_train, X_val, X_test):
    X_train.to_csv(FEATURE_DIR / f"X_train_{name}.csv", index=False)
    X_val.to_csv(FEATURE_DIR / f"X_val_{name}.csv", index=False)
    X_test.to_csv(FEATURE_DIR / f"X_test_{name}.csv", index=False)

    print(f"Saved feature set: {name}")
    print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)


save_feature_set("basic_gc_cpg_skew_bins_v1", X_train_basic, X_val_basic, X_test_basic)
save_feature_set("k3_plus_basic_v1", X_train_k3, X_val_k3, X_test_k3)
save_feature_set("k4_plus_basic_v1", X_train_k4, X_val_k4, X_test_k4)
save_feature_set("k3_k4_plus_basic_v1", X_train_main, X_val_main, X_test_main)

if RUN_K5:
    save_feature_set("k3_k4_k5_plus_basic_v1", X_train_k345, X_val_k345, X_test_k345)

# Save labels and metadata
y_train = train_df[LABEL_COL].astype(int)
y_val = val_df[LABEL_COL].astype(int)
y_test = test_df[LABEL_COL].astype(int)

train_df[["gene_id", "gene_symbol", "label"]].to_csv(
    FEATURE_DIR / "train_genomic_metadata_v1.csv",
    index=False
)

val_df[["gene_id", "gene_symbol", "label"]].to_csv(
    FEATURE_DIR / "val_genomic_metadata_v1.csv",
    index=False
)

test_df[["gene_id", "gene_symbol", "label"]].to_csv(
    FEATURE_DIR / "test_genomic_metadata_v1.csv",
    index=False
)

Saved feature set: basic_gc_cpg_skew_bins_v1
Train: (1264, 36) Val: (271, 36) Test: (271, 36)
Saved feature set: k3_plus_basic_v1
Train: (1264, 100) Val: (271, 100) Test: (271, 100)
Saved feature set: k4_plus_basic_v1
Train: (1264, 292) Val: (271, 292) Test: (271, 292)
Saved feature set: k3_k4_plus_basic_v1
Train: (1264, 356) Val: (271, 356) Test: (271, 356)
Saved feature set: k3_k4_k5_plus_basic_v1
Train: (1264, 1380) Val: (271, 1380) Test: (271, 1380)


In [16]:
# ============================================================
# FEATURE QC SUMMARY
# ============================================================

feature_sets = {
    "Basic GC/CpG/skew/bins": (X_train_basic, X_val_basic, X_test_basic),
    "K3 + Basic": (X_train_k3, X_val_k3, X_test_k3),
    "K4 + Basic": (X_train_k4, X_val_k4, X_test_k4),
    "K3 + K4 + Basic": (X_train_main, X_val_main, X_test_main),
}

if RUN_K5:
    feature_sets["K3 + K4 + K5 + Basic"] = (X_train_k345, X_val_k345, X_test_k345)

feature_qc_records = []

for name, (Xtr, Xv, Xte) in feature_sets.items():
    feature_qc_records.append({
        "feature_set": name,
        "n_train": Xtr.shape[0],
        "n_val": Xv.shape[0],
        "n_test": Xte.shape[0],
        "n_features": Xtr.shape[1],
        "train_missing_values": int(Xtr.isna().sum().sum()),
        "val_missing_values": int(Xv.isna().sum().sum()),
        "test_missing_values": int(Xte.isna().sum().sum()),
        "train_inf_values": int(np.isinf(Xtr.values).sum()),
        "val_inf_values": int(np.isinf(Xv.values).sum()),
        "test_inf_values": int(np.isinf(Xte.values).sum()),
    })

feature_qc_df = pd.DataFrame(feature_qc_records)

display(feature_qc_df)

feature_qc_df.to_csv(
    RESULT_DIR / "phase2_1_feature_qc_summary_v1.csv",
    index=False
)

,feature_set,n_train,n_val,n_test,n_features,train_missing_values,val_missing_values,test_missing_values,train_inf_values,val_inf_values,test_inf_values
0,Basic GC/CpG/skew/bins,1264,271,271,36,0,0,0,0,0,0
1,K3 + Basic,1264,271,271,100,0,0,0,0,0,0
2,K4 + Basic,1264,271,271,292,0,0,0,0,0,0
3,K3 + K4 + Basic,1264,271,271,356,0,0,0,0,0,0
4,K3 + K4 + K5 + Basic,1264,271,271,1380,0,0,0,0,0,0


In [17]:
# ============================================================
# EVALUATION HELPERS
# ============================================================

def get_positive_class_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    if hasattr(model, "decision_function"):
        return model.decision_function(X)

    return model.predict(X)


def evaluate_binary_classifier(model, X, y, model_name, dataset_name, feature_set):
    y_pred = model.predict(X)
    y_score = get_positive_class_score(model, X)

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "feature_set": feature_set,
        "model": model_name,
        "dataset": dataset_name,

        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y, y_pred, zero_division=0),

        "roc_auc": roc_auc_score(y, y_score),
        "pr_auc": average_precision_score(y, y_score),
        "mcc": matthews_corrcoef(y, y_pred),

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

In [18]:
# ============================================================
# CV + SCORING
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "mcc": make_scorer(matthews_corrcoef)
}

In [19]:
# ============================================================
# MODEL DEFINITIONS
# ============================================================

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False


def build_baseline_models():
    models = {
        "Logistic Regression": Pipeline([
            ("variance", VarianceThreshold(threshold=0.0)),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=5000,
                random_state=42
            ))
        ]),

        "SVM RBF": Pipeline([
            ("variance", VarianceThreshold(threshold=0.0)),
            ("scaler", StandardScaler()),
            ("model", SVC(
                kernel="rbf",
                probability=True,
                random_state=42
            ))
        ]),

        "Random Forest": Pipeline([
            ("variance", VarianceThreshold(threshold=0.0)),
            ("model", RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            ))
        ])
    }

    if LIGHTGBM_AVAILABLE:
        models["LightGBM"] = Pipeline([
            ("variance", VarianceThreshold(threshold=0.0)),
            ("model", LGBMClassifier(
                n_estimators=300,
                learning_rate=0.05,
                num_leaves=15,
                random_state=42,
                n_jobs=-1,
                verbose=-1
            ))
        ])
    else:
        models["Hist Gradient Boosting"] = Pipeline([
            ("variance", VarianceThreshold(threshold=0.0)),
            ("model", HistGradientBoostingClassifier(
                learning_rate=0.05,
                max_iter=300,
                random_state=42
            ))
        ])

    return models

In [20]:
# ============================================================
# FEATURE SET ABLATION WITH BASELINE CV
# ============================================================

ablation_feature_sets = {
    "Basic GC/CpG/skew/bins": X_train_basic,
    "K3 + Basic": X_train_k3,
    "K4 + Basic": X_train_k4,
    "K3 + K4 + Basic": X_train_main,
}

if RUN_K5:
    ablation_feature_sets["K3 + K4 + K5 + Basic"] = X_train_k345


ablation_cv_results = []

for feature_set_name, X_train_fs in ablation_feature_sets.items():
    print("=" * 100)
    print("Feature set:", feature_set_name, "| shape:", X_train_fs.shape)

    models = build_baseline_models()

    for model_name, pipeline in models.items():
        print("- CV:", model_name)

        cv_output = cross_validate(
            estimator=pipeline,
            X=X_train_fs,
            y=y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
            return_train_score=True
        )

        row = {
            "feature_set": feature_set_name,
            "n_features": X_train_fs.shape[1],
            "model": model_name,

            "train_roc_auc_mean": cv_output["train_roc_auc"].mean(),
            "train_roc_auc_std": cv_output["train_roc_auc"].std(),

            "cv_roc_auc_mean": cv_output["test_roc_auc"].mean(),
            "cv_roc_auc_std": cv_output["test_roc_auc"].std(),

            "cv_f1_mean": cv_output["test_f1"].mean(),
            "cv_f1_std": cv_output["test_f1"].std(),

            "cv_pr_auc_mean": cv_output["test_pr_auc"].mean(),
            "cv_pr_auc_std": cv_output["test_pr_auc"].std(),

            "cv_mcc_mean": cv_output["test_mcc"].mean(),
            "cv_mcc_std": cv_output["test_mcc"].std(),

            "overfit_gap_train_minus_cv": (
                cv_output["train_roc_auc"].mean() - cv_output["test_roc_auc"].mean()
            )
        }

        ablation_cv_results.append(row)

ablation_cv_df = pd.DataFrame(ablation_cv_results).sort_values(
    by=["cv_roc_auc_mean", "cv_mcc_mean"],
    ascending=False
)

display(ablation_cv_df)

ablation_cv_df.to_csv(
    RESULT_DIR / "phase2_1_genomic_feature_ablation_5fold_cv_v1.csv",
    index=False
)

Feature set: Basic GC/CpG/skew/bins | shape: (1264, 36)
- CV: Logistic Regression
- CV: SVM RBF
- CV: Random Forest
- CV: LightGBM
Feature set: K3 + Basic | shape: (1264, 100)
- CV: Logistic Regression
- CV: SVM RBF
- CV: Random Forest
- CV: LightGBM
Feature set: K4 + Basic | shape: (1264, 292)
- CV: Logistic Regression
- CV: SVM RBF
- CV: Random Forest
- CV: LightGBM
Feature set: K3 + K4 + Basic | shape: (1264, 356)
- CV: Logistic Regression
- CV: SVM RBF
- CV: Random Forest
- CV: LightGBM
Feature set: K3 + K4 + K5 + Basic | shape: (1264, 1380)
- CV: Logistic Regression
- CV: SVM RBF
- CV: Random Forest
- CV: LightGBM


,feature_set,n_features,model,train_roc_auc_mean,train_roc_auc_std,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_f1_std,cv_pr_auc_mean,cv_pr_auc_std,cv_mcc_mean,cv_mcc_std,overfit_gap_train_minus_cv
17,K3 + K4 + K5 + Basic,1380,SVM RBF,0.993841,8.086168e-04,0.624422,0.021113,0.575379,0.034721,0.624723,0.023956,0.174859,0.049384,0.369418
18,K3 + K4 + K5 + Basic,1380,Random Forest,1.000000,7.021667e-17,0.622458,0.016496,0.583233,0.032629,0.610620,0.017700,0.187261,0.040197,0.377542
13,K3 + K4 + Basic,356,SVM RBF,0.952103,2.197970e-03,0.621576,0.019897,0.581153,0.011701,0.619110,0.026693,0.182445,0.014757,0.330527
9,K4 + Basic,292,SVM RBF,0.960187,1.849461e-03,0.620770,0.018537,0.579116,0.014679,0.620640,0.024589,0.181214,0.016887,0.339418
19,K3 + K4 + K5 + Basic,1380,LightGBM,1.000000,0.000000e+00,0.618343,0.027942,0.575254,0.018136,0.615839,0.027766,0.171244,0.032488,0.381657
11,K4 + Basic,292,LightGBM,1.000000,0.000000e+00,0.614648,0.035414,0.575325,0.032364,0.613654,0.021553,0.155210,0.060339,0.385352
10,K4 + Basic,292,Random Forest,1.000000,4.965068e-17,0.613409,0.014406,0.575292,0.032567,0.615503,0.020313,0.166652,0.050092,0.386591
15,K3 + K4 + Basic,356,LightGBM,1.000000,0.000000e+00,0.612571,0.028715,0.571455,0.027371,0.610557,0.015305,0.145662,0.056358,0.387429
14,K3 + K4 + Basic,356,Random Forest,1.000000,0.000000e+00,0.611158,0.009325,0.576099,0.023497,0.614920,0.026162,0.174948,0.021226,0.388842
4,K3 + Basic,100,Logistic Regression,0.705615,2.065095e-03,0.610516,0.017772,0.565969,0.032724,0.608493,0.016032,0.152661,0.041210,0.095099


In [21]:
# ============================================================
# SELECT MAIN FEATURE SET
# ============================================================

MAIN_FEATURE_SET_NAME = "K3 + K4 + Basic"

X_train_main_selected = X_train_main.copy()
X_val_main_selected = X_val_main.copy()
X_test_main_selected = X_test_main.copy()

# Optional switch to K5 if ablation clearly supports it.
# MAIN_FEATURE_SET_NAME = "K3 + K4 + K5 + Basic"
# X_train_main_selected = X_train_k345.copy()
# X_val_main_selected = X_val_k345.copy()
# X_test_main_selected = X_test_k345.copy()

print("Selected main feature set:", MAIN_FEATURE_SET_NAME)
print("Train:", X_train_main_selected.shape)
print("Val:", X_val_main_selected.shape)
print("Test:", X_test_main_selected.shape)

Selected main feature set: K3 + K4 + Basic
Train: (1264, 356)
Val: (271, 356)
Test: (271, 356)


In [22]:
# ============================================================
# PARAMETER GRIDS
# ============================================================

genomic_models = build_baseline_models()

genomic_param_grids = {
    "Logistic Regression": {
        "model__C": [0.001, 0.003, 0.01, 0.03, 0.1],
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs"]
    },

    "SVM RBF": {
        "model__C": [0.1, 1, 10],
        "model__gamma": [0.001, 0.01, "scale"]
    },

    "Random Forest": {
        "model__n_estimators": [300, 500],
        "model__max_depth": [5, 8, 10, None],
        "model__min_samples_leaf": [5, 10, 20],
        "model__max_features": ["sqrt", 0.2],
        "model__bootstrap": [True]
    }
}

if LIGHTGBM_AVAILABLE:
    genomic_param_grids["LightGBM"] = {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.03, 0.05],
        "model__num_leaves": [7, 15],
        "model__max_depth": [3, 5],
        "model__min_child_samples": [20, 50],
        "model__subsample": [0.8],
        "model__colsample_bytree": [0.8, 1.0],
        "model__reg_alpha": [0.1],
        "model__reg_lambda": [1, 5]
    }
else:
    genomic_param_grids["Hist Gradient Boosting"] = {
        "model__learning_rate": [0.03, 0.05],
        "model__max_iter": [100, 200],
        "model__max_leaf_nodes": [15, 31],
        "model__min_samples_leaf": [20, 50]
    }

In [23]:
# ============================================================
# RUN GRIDSEARCHCV
# ============================================================

genomic_grid_search_results = []
genomic_best_estimators = {}

for model_name, pipeline in genomic_models.items():
    print("=" * 100)
    print("GridSearchCV:", model_name)

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=genomic_param_grids[model_name],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
        verbose=1
    )

    grid.fit(X_train_main_selected, y_train)

    genomic_best_estimators[model_name] = grid.best_estimator_

    best_idx = grid.best_index_

    result_row = {
        "feature_set": MAIN_FEATURE_SET_NAME,
        "model": model_name,
        "best_cv_roc_auc": grid.best_score_,
        "best_train_roc_auc": grid.cv_results_["mean_train_score"][best_idx],
        "overfit_gap_train_minus_cv": (
            grid.cv_results_["mean_train_score"][best_idx] - grid.best_score_
        ),
        "best_params": grid.best_params_
    }

    genomic_grid_search_results.append(result_row)

    print("Best CV ROC-AUC:", grid.best_score_)
    print("Best train ROC-AUC:", grid.cv_results_["mean_train_score"][best_idx])
    print("Gap:", result_row["overfit_gap_train_minus_cv"])
    print("Best params:", grid.best_params_)

genomic_grid_results_df = pd.DataFrame(genomic_grid_search_results).sort_values(
    by="best_cv_roc_auc",
    ascending=False
)

display(genomic_grid_results_df)

genomic_grid_results_df.to_csv(
    RESULT_DIR / "phase2_1_genomic_gridsearch_results_v1.csv",
    index=False
)

GridSearchCV: Logistic Regression
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best CV ROC-AUC: 0.6379244261134025
Best train ROC-AUC: 0.7285449400757227
Gap: 0.09062051396232018
Best params: {'model__C': 0.001, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
GridSearchCV: SVM RBF
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best CV ROC-AUC: 0.6285777769842261
Best train ROC-AUC: 0.8095509990937227
Gap: 0.18097322210949662
Best params: {'model__C': 1, 'model__gamma': 0.001}
GridSearchCV: Random Forest
Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best CV ROC-AUC: 0.6277841261905753
Best train ROC-AUC: 0.9987378971614295
Gap: 0.37095377097085414
Best params: {'model__bootstrap': True, 'model__max_depth': 10, 'model__max_features': 0.2, 'model__min_samples_leaf': 10, 'model__n_estimators': 300}
GridSearchCV: LightGBM
Fitting 5 folds for each of 128 candidates, totalling 640 fits
Best CV ROC-AUC: 0.6389664585577597
Best train ROC-AUC: 0.9801

,feature_set,model,best_cv_roc_auc,best_train_roc_auc,overfit_gap_train_minus_cv,best_params
3,K3 + K4 + Basic,LightGBM,0.638966,0.980187,0.341220,"{'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__min_child_samples': 50..."
0,K3 + K4 + Basic,Logistic Regression,0.637924,0.728545,0.090621,"{'model__C': 0.001, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}"
1,K3 + K4 + Basic,SVM RBF,0.628578,0.809551,0.180973,"{'model__C': 1, 'model__gamma': 0.001}"
2,K3 + K4 + Basic,Random Forest,0.627784,0.998738,0.370954,"{'model__bootstrap': True, 'model__max_depth': 10, 'model__max_features': 0.2, 'model__min_samples_leaf': 10, 'model..."


In [24]:
# ============================================================
# TRAIN + VALIDATION EVALUATION
# ============================================================

genomic_tuned_eval_records = []

for model_name, model in genomic_best_estimators.items():
    print("=" * 100)
    print("Evaluating:", model_name)

    train_metrics = evaluate_binary_classifier(
        model=model,
        X=X_train_main_selected,
        y=y_train,
        model_name=model_name,
        dataset_name="train",
        feature_set=MAIN_FEATURE_SET_NAME
    )

    val_metrics = evaluate_binary_classifier(
        model=model,
        X=X_val_main_selected,
        y=y_val,
        model_name=model_name,
        dataset_name="validation",
        feature_set=MAIN_FEATURE_SET_NAME
    )

    genomic_tuned_eval_records.extend([train_metrics, val_metrics])

genomic_tuned_eval_df = pd.DataFrame(genomic_tuned_eval_records)

display(genomic_tuned_eval_df)

genomic_validation_comparison = genomic_tuned_eval_df[
    genomic_tuned_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(genomic_validation_comparison)

genomic_tuned_eval_df.to_csv(
    RESULT_DIR / "phase2_1_genomic_tuned_train_validation_metrics_v1.csv",
    index=False
)

genomic_validation_comparison.to_csv(
    RESULT_DIR / "phase2_1_genomic_validation_comparison_v1.csv",
    index=False
)

Evaluating: Logistic Regression
Evaluating: SVM RBF
Evaluating: Random Forest
Evaluating: LightGBM


,feature_set,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,K3 + K4 + Basic,Logistic Regression,train,0.662975,0.672241,0.636076,0.689873,0.653659,0.720861,0.708135,0.326422,436,196,230,402
1,K3 + K4 + Basic,Logistic Regression,validation,0.619926,0.624060,0.610294,0.629630,0.617100,0.653540,0.673381,0.239963,85,50,53,83
2,K3 + K4 + Basic,SVM RBF,train,0.721519,0.728758,0.705696,0.737342,0.717042,0.800421,0.791594,0.443260,466,166,186,446
3,K3 + K4 + Basic,SVM RBF,validation,0.594096,0.597015,0.588235,0.600000,0.592593,0.649918,0.669187,0.188246,81,54,56,80
4,K3 + K4 + Basic,Random Forest,train,0.981013,0.973520,0.988924,0.973101,0.981162,0.998981,0.999005,0.962146,615,17,7,625
5,K3 + K4 + Basic,Random Forest,validation,0.627306,0.644628,0.573529,0.681481,0.607004,0.670752,0.704442,0.256482,92,43,58,78
6,K3 + K4 + Basic,LightGBM,train,0.912975,0.912975,0.912975,0.912975,0.912975,0.969741,0.968471,0.825949,577,55,55,577
7,K3 + K4 + Basic,LightGBM,validation,0.594096,0.595588,0.595588,0.592593,0.595588,0.632898,0.674220,0.188181,80,55,55,81


,feature_set,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
5,K3 + K4 + Basic,Random Forest,validation,0.627306,0.644628,0.573529,0.681481,0.607004,0.670752,0.704442,0.256482,92,43,58,78
1,K3 + K4 + Basic,Logistic Regression,validation,0.619926,0.624060,0.610294,0.629630,0.617100,0.653540,0.673381,0.239963,85,50,53,83
3,K3 + K4 + Basic,SVM RBF,validation,0.594096,0.597015,0.588235,0.600000,0.592593,0.649918,0.669187,0.188246,81,54,56,80
7,K3 + K4 + Basic,LightGBM,validation,0.594096,0.595588,0.595588,0.592593,0.595588,0.632898,0.674220,0.188181,80,55,55,81


In [25]:
# ============================================================
# SOFT VOTING
# ============================================================

genomic_voting_estimators = []

for model_name, estimator in genomic_best_estimators.items():
    short_name = model_name.lower().replace(" ", "_").replace("-", "_")
    genomic_voting_estimators.append((short_name, estimator))

genomic_soft_voting = VotingClassifier(
    estimators=genomic_voting_estimators,
    voting="soft",
    n_jobs=-1
)

genomic_soft_voting.fit(X_train_main_selected, y_train)

voting_train_metrics = evaluate_binary_classifier(
    genomic_soft_voting,
    X_train_main_selected,
    y_train,
    "Soft Voting",
    "train",
    MAIN_FEATURE_SET_NAME
)

voting_val_metrics = evaluate_binary_classifier(
    genomic_soft_voting,
    X_val_main_selected,
    y_val,
    "Soft Voting",
    "validation",
    MAIN_FEATURE_SET_NAME
)

display(pd.DataFrame([voting_train_metrics, voting_val_metrics]))

genomic_best_estimators["Soft Voting"] = genomic_soft_voting

,feature_set,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,K3 + K4 + Basic,Soft Voting,train,0.863924,0.870968,0.854430,0.873418,0.86262,0.938802,0.937881,0.727979,552,80,92,540
1,K3 + K4 + Basic,Soft Voting,validation,0.619926,0.624060,0.610294,0.629630,0.61710,0.658061,0.693627,0.239963,85,50,53,83


In [26]:
# ============================================================
# STACKING
# ============================================================

genomic_stacking_estimators = []

for model_name, estimator in genomic_best_estimators.items():
    if model_name == "Soft Voting":
        continue

    short_name = model_name.lower().replace(" ", "_").replace("-", "_")
    genomic_stacking_estimators.append((short_name, estimator))

genomic_stacking = StackingClassifier(
    estimators=genomic_stacking_estimators,
    final_estimator=LogisticRegression(
        max_iter=5000,
        C=1.0,
        random_state=42
    ),
    cv=5,
    stack_method="predict_proba",
    n_jobs=-1,
    passthrough=False
)

genomic_stacking.fit(X_train_main_selected, y_train)

stacking_train_metrics = evaluate_binary_classifier(
    genomic_stacking,
    X_train_main_selected,
    y_train,
    "Stacking",
    "train",
    MAIN_FEATURE_SET_NAME
)

stacking_val_metrics = evaluate_binary_classifier(
    genomic_stacking,
    X_val_main_selected,
    y_val,
    "Stacking",
    "validation",
    MAIN_FEATURE_SET_NAME
)

display(pd.DataFrame([stacking_train_metrics, stacking_val_metrics]))

genomic_best_estimators["Stacking"] = genomic_stacking

,feature_set,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,K3 + K4 + Basic,Stacking,train,0.853639,0.861066,0.843354,0.863924,0.852118,0.931444,0.930360,0.707428,546,86,99,533
1,K3 + K4 + Basic,Stacking,validation,0.616236,0.617647,0.617647,0.614815,0.617647,0.659205,0.691735,0.232462,83,52,52,84


In [27]:
# ============================================================
# FINAL VALIDATION COMPARISON
# ============================================================

genomic_all_eval_df = pd.concat(
    [
        genomic_tuned_eval_df,
        pd.DataFrame([
            voting_train_metrics,
            voting_val_metrics,
            stacking_train_metrics,
            stacking_val_metrics
        ])
    ],
    ignore_index=True
)

genomic_all_validation_results = genomic_all_eval_df[
    genomic_all_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(genomic_all_validation_results)

genomic_all_eval_df.to_csv(
    RESULT_DIR / "phase2_1_genomic_all_train_validation_metrics_v1.csv",
    index=False
)

genomic_all_validation_results.to_csv(
    RESULT_DIR / "phase2_1_genomic_all_validation_comparison_v1.csv",
    index=False
)

,feature_set,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
5,K3 + K4 + Basic,Random Forest,validation,0.627306,0.644628,0.573529,0.681481,0.607004,0.670752,0.704442,0.256482,92,43,58,78
11,K3 + K4 + Basic,Stacking,validation,0.616236,0.617647,0.617647,0.614815,0.617647,0.659205,0.691735,0.232462,83,52,52,84
9,K3 + K4 + Basic,Soft Voting,validation,0.619926,0.624060,0.610294,0.629630,0.617100,0.658061,0.693627,0.239963,85,50,53,83
1,K3 + K4 + Basic,Logistic Regression,validation,0.619926,0.624060,0.610294,0.629630,0.617100,0.653540,0.673381,0.239963,85,50,53,83
3,K3 + K4 + Basic,SVM RBF,validation,0.594096,0.597015,0.588235,0.600000,0.592593,0.649918,0.669187,0.188246,81,54,56,80
7,K3 + K4 + Basic,LightGBM,validation,0.594096,0.595588,0.595588,0.592593,0.595588,0.632898,0.674220,0.188181,80,55,55,81


In [28]:
# ============================================================
# SELECT FINAL MODEL FROM VALIDATION ROC-AUC
# ============================================================

best_validation_row = genomic_all_validation_results.iloc[0]
final_genomic_model_name = best_validation_row["model"]
final_genomic_model = genomic_best_estimators[final_genomic_model_name]

print("Final selected genomic model:", final_genomic_model_name)
display(best_validation_row)

pd.DataFrame([best_validation_row]).to_csv(
    RESULT_DIR / "phase2_1_genomic_final_model_validation_summary_v1.csv",
    index=False
)

Final selected genomic model: Random Forest


,5
feature_set,K3 + K4 + Basic
model,Random Forest
dataset,validation
accuracy,0.627306
precision,0.644628
recall_sensitivity,0.573529
specificity,0.681481
f1,0.607004
roc_auc,0.670752
pr_auc,0.704442


In [29]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================

genomic_final_test_metrics = evaluate_binary_classifier(
    model=final_genomic_model,
    X=X_test_main_selected,
    y=y_test,
    model_name=final_genomic_model_name,
    dataset_name="test",
    feature_set=MAIN_FEATURE_SET_NAME
)

genomic_final_test_metrics_df = pd.DataFrame([genomic_final_test_metrics])

display(genomic_final_test_metrics_df)

genomic_final_test_metrics_df.to_csv(
    RESULT_DIR / "phase2_1_genomic_final_test_metrics_v1.csv",
    index=False
)

,feature_set,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,K3 + K4 + Basic,Random Forest,test,0.627306,0.616438,0.666667,0.588235,0.640569,0.649619,0.632732,0.255669,80,56,45,90


In [30]:
# ============================================================
# DIAGNOSTIC TEST ALL MODELS
# Not used for model selection.
# ============================================================

genomic_diagnostic_test_records = []

for model_name, model in genomic_best_estimators.items():
    metrics = evaluate_binary_classifier(
        model=model,
        X=X_test_main_selected,
        y=y_test,
        model_name=model_name,
        dataset_name="test_diagnostic",
        feature_set=MAIN_FEATURE_SET_NAME
    )

    genomic_diagnostic_test_records.append(metrics)

genomic_diagnostic_test_df = pd.DataFrame(genomic_diagnostic_test_records).sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(genomic_diagnostic_test_df)

genomic_diagnostic_test_df.to_csv(
    RESULT_DIR / "phase2_1_genomic_diagnostic_all_models_test_metrics_v1.csv",
    index=False
)

,feature_set,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
1,K3 + K4 + Basic,SVM RBF,test_diagnostic,0.645756,0.638298,0.666667,0.625000,0.652174,0.676471,0.650405,0.291905,85,51,45,90
4,K3 + K4 + Basic,Soft Voting,test_diagnostic,0.630996,0.622378,0.659259,0.602941,0.640288,0.672004,0.648989,0.262601,82,54,46,89
5,K3 + K4 + Basic,Stacking,test_diagnostic,0.630996,0.622378,0.659259,0.602941,0.640288,0.671514,0.650205,0.262601,82,54,46,89
0,K3 + K4 + Basic,Logistic Regression,test_diagnostic,0.616236,0.608392,0.644444,0.588235,0.625899,0.659259,0.647477,0.233035,80,56,48,87
2,K3 + K4 + Basic,Random Forest,test_diagnostic,0.627306,0.616438,0.666667,0.588235,0.640569,0.649619,0.632732,0.255669,80,56,45,90
3,K3 + K4 + Basic,LightGBM,test_diagnostic,0.594096,0.583893,0.644444,0.544118,0.612676,0.649401,0.623016,0.189504,74,62,48,87


In [31]:
# ============================================================
# SAVE FINAL TEST PREDICTIONS
# ============================================================

y_test_score = get_positive_class_score(final_genomic_model, X_test_main_selected)
y_test_pred = final_genomic_model.predict(X_test_main_selected)

genomic_test_predictions_df = test_df[[
    "gene_id",
    "gene_symbol",
    "label",
    "dataset_role"
]].copy()

genomic_test_predictions_df["true_label"] = y_test.values
genomic_test_predictions_df["pred_label"] = y_test_pred
genomic_test_predictions_df["pred_score_t2d_associated"] = y_test_score
genomic_test_predictions_df["feature_set"] = MAIN_FEATURE_SET_NAME
genomic_test_predictions_df["model"] = final_genomic_model_name

display(genomic_test_predictions_df.head())

genomic_test_predictions_df.to_csv(
    RESULT_DIR / "phase2_1_genomic_final_test_predictions_v1.csv",
    index=False
)

,gene_id,gene_symbol,label,dataset_role,true_label,pred_label,pred_score_t2d_associated,feature_set,model
0,ENSG00000101639,CEP192,1,positive,1,0,0.452821,K3 + K4 + Basic,Random Forest
1,ENSG00000198793,MTOR,1,positive,1,0,0.386692,K3 + K4 + Basic,Random Forest
2,ENSG00000064607,SUGP2,0,negative,0,0,0.490883,K3 + K4 + Basic,Random Forest
3,ENSG00000180332,KCTD4,0,negative,0,0,0.460903,K3 + K4 + Basic,Random Forest
4,ENSG00000166226,CCT2,0,negative,0,0,0.392621,K3 + K4 + Basic,Random Forest


In [32]:
# ============================================================
# SAVE MODELS
# ============================================================

for model_name, model in genomic_best_estimators.items():
    safe_name = model_name.lower().replace(" ", "_").replace("-", "_")

    model_path = MODEL_DIR / f"phase2_1_genomic_{safe_name}_best_estimator_v1.pkl"
    joblib.dump(model, model_path)

    print("Saved:", model_path)

final_model_path = MODEL_DIR / "phase2_1_genomic_final_selected_model_v1.pkl"
joblib.dump(final_genomic_model, final_model_path)

print("Final genomic model saved:", final_model_path)

Saved: /content/drive/MyDrive/Project_Protein/model/phase2_genomic_regulatory_baseline/models/phase2_1_genomic_logistic_regression_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase2_genomic_regulatory_baseline/models/phase2_1_genomic_svm_rbf_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase2_genomic_regulatory_baseline/models/phase2_1_genomic_random_forest_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase2_genomic_regulatory_baseline/models/phase2_1_genomic_lightgbm_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase2_genomic_regulatory_baseline/models/phase2_1_genomic_soft_voting_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase2_genomic_regulatory_baseline/models/phase2_1_genomic_stacking_best_estimator_v1.pkl
Final genomic model saved: /content/drive/MyDrive/Project_Protein/model/phase2_genomic_regulatory_baseline/models/phase2_1_genomic

In [33]:
# ============================================================
# PROTEIN VS GENOMIC COMPARISON
# ============================================================

protein_vs_genomic_df = pd.DataFrame([
    {
        "branch": "Protein",
        "best_representation": "ProtBERT sliding-window",
        "best_model": "Logistic Regression",
        "test_roc_auc": 0.6487,
        "test_pr_auc": 0.6551,
        "test_f1": 0.5896,
        "test_mcc": 0.1941,
        "note": "Best ranking protein representation at default threshold"
    },
    {
        "branch": "Genomic regulatory",
        "best_representation": MAIN_FEATURE_SET_NAME,
        "best_model": final_genomic_model_name,
        "test_roc_auc": genomic_final_test_metrics["roc_auc"],
        "test_pr_auc": genomic_final_test_metrics["pr_auc"],
        "test_f1": genomic_final_test_metrics["f1"],
        "test_mcc": genomic_final_test_metrics["mcc"],
        "note": "Phase 2.1 handcrafted genomic baseline"
    }
])

display(protein_vs_genomic_df)

protein_vs_genomic_df.to_csv(
    RESULT_DIR / "phase2_1_protein_vs_genomic_comparison_v1.csv",
    index=False
)

,branch,best_representation,best_model,test_roc_auc,test_pr_auc,test_f1,test_mcc,note
0,Protein,ProtBERT sliding-window,Logistic Regression,0.648700,0.655100,0.589600,0.194100,Best ranking protein representation at default threshold
1,Genomic regulatory,K3 + K4 + Basic,Random Forest,0.649619,0.632732,0.640569,0.255669,Phase 2.1 handcrafted genomic baseline
